# Human-in-the-Loop Approval for Irreversible Agent Actions

**Addresses:** [anthropic-cookbook issue #701](https://github.com/anthropics/anthropic-cookbook/issues/701)

## The problem with naive approval gates

Most tutorials implement human-in-the-loop approval as a simple yes/no prompt:

```
Agent: "About to delete 50,000 records. Approve? [y/n]"
Human: y
```

This is harmful. The human cannot assess risk without knowing:
- **What specifically** will happen (not a vague summary)
- **Whether rollback is possible** — and if so, how
- **The blast radius** — how many records, users, dollars are at stake
- **Whether a dry-run was run first** to catch errors before real execution

When the gate skips this preflight, users rubber-stamp actions they don't understand. The approval becomes theater.

## The correct pattern

**Ask for approval only after verifying rollback capability.** The sequence is:

1. Agent identifies the action as potentially irreversible
2. Agent classifies reversibility: `reversible` / `recoverable` / `permanent`
3. For `permanent` actions: runs a dry-run, estimates impact, checks backup state
4. Only then presents a structured approval request with all findings attached
5. Times out and **aborts** (safe default) rather than proceeding on silence

This notebook builds that pattern step by step.

## Setup

In [ ]:
%pip install anthropic python-dotenv --quiet

In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Optional

import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL_NAME = "claude-sonnet-4-6"

print(f"Anthropic client ready. Model: {MODEL_NAME}")

Anthropic client ready. Model: claude-sonnet-4-6


## Part 1 — The naive pattern (and why it fails)

Here is the pattern most tutorials recommend. Read it critically.

In [ ]:
# ============================================================
# NAIVE PATTERN — shown here only to illustrate what NOT to do
# ============================================================

def naive_approval_gate(action_name: str) -> bool:
    """The naive pattern: ask before acting, with no context.

    WARNING: Do not use this in production. Shown for comparison only.
    """
    response = input(f"Action: {action_name}\nApprove? [y/n]: ")
    return response.strip().lower() == "y"


print("=== NAIVE APPROVAL GATE (DO NOT USE) ===")
print()
print("Action: delete_user_records")
print("Approve? [y/n]: ")
print()
print("--- Why this fails ---")
problems = [
    ("No specifics",
     "'delete_user_records' tells the human nothing.\n"
     "           How many records? Which users? Is there a backup?"),
    ("No rollback info",
     "The human doesn't know if this can be undone.\n"
     "           They're approving in the dark."),
    ("No blast radius",
     "'$cost', 'N users affected' — all unknown."),
    ("No dry-run",
     "If the agent has a bug in its WHERE clause,\n"
     "           you discover that AFTER deleting production data."),
    ("On timeout, what happens?",
     "Most naive gates block forever\n"
     "           or proceed anyway — both are wrong."),
]
for i, (label, detail) in enumerate(problems, 1):
    print(f"Problem {i}: {label}. {detail}")
    print()
print("Result: The human clicks 'y' because they assume the agent checked.")
print("        The agent assumed the human checked. Nobody checked.")

=== NAIVE APPROVAL GATE (DO NOT USE) ===

Action: delete_user_records
Approve? [y/n]: 

--- Why this fails ---
Problem 1: No specifics. 'delete_user_records' tells the human nothing.
           How many records? Which users? Is there a backup?

Problem 2: No rollback info. The human doesn't know if this can be undone.
           They're approving in the dark.

Problem 3: No blast radius. '$cost', 'N users affected' — all unknown.

Problem 4: No dry-run. If the agent has a bug in its WHERE clause,
           you discover that AFTER deleting production data.

Problem 5: On timeout, what happens? Most naive gates block forever
           or proceed anyway — both are wrong.

Result: The human clicks 'y' because they assume the agent checked.
        The agent assumed the human checked. Nobody checked.


## Part 2 — What the approval request MUST include

A well-formed approval request answers four questions before asking anything:

| Field | What it tells the human |
|---|---|
| `action_description` | What will happen, specifically (not vaguely) |
| `irreversible_effects` | What cannot be undone, even if rollback exists |
| `blast_radius` | Records affected, users impacted, cost at risk |
| `dry_run_result` | Whether a staging/preview run was completed first |
| `rollback_path` | Exact steps to undo, or `None` if permanent |
| `timeout_behavior` | What happens if no answer arrives — always `ABORT` |

The human should be able to make an informed decision in under 30 seconds. If your approval request requires them to go check something else first, the agent did not do enough preflight work.

## Part 3 — Building the `IrreversibleActionGate`

### 3a. Data types

In [ ]:
class Reversibility(str, Enum):
    """Classification of an action's reversibility."""

    REVERSIBLE = "reversible"      # Undo is trivial and instant (e.g., rename a file)
    RECOVERABLE = "recoverable"    # Rollback exists but requires explicit steps (e.g., restore from backup)
    PERMANENT = "permanent"        # No undo path; action is final (e.g., send email, purge data)


@dataclass
class BlastRadius:
    """Quantified impact of an action."""

    records_affected: int = 0
    users_impacted: int = 0
    estimated_cost_usd: float = 0.0
    description: str = ""

    def summary(self) -> str:
        parts = []
        if self.records_affected:
            parts.append(f"{self.records_affected:,} records")
        if self.users_impacted:
            parts.append(f"{self.users_impacted:,} users")
        if self.estimated_cost_usd:
            parts.append(f"${self.estimated_cost_usd:,.2f} at risk")
        return "; ".join(parts) if parts else "unknown scope"


@dataclass
class ApprovalRequest:
    """Structured approval packet shown to the human before any irreversible action."""

    action_name: str
    action_description: str           # Specific: what exactly will execute
    reversibility: Reversibility
    irreversible_effects: list[str]   # What CANNOT be undone even with rollback
    blast_radius: BlastRadius
    dry_run_completed: bool
    dry_run_summary: Optional[str]
    rollback_path: Optional[str]      # None = no rollback exists
    timeout_seconds: int = 60
    requested_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

    def render(self) -> str:
        """Format the approval request for human display."""
        lines = [
            "=" * 60,
            "  APPROVAL REQUIRED — IRREVERSIBLE ACTION",
            "=" * 60,
            f"  Action      : {self.action_name}",
            f"  What happens: {self.action_description}",
            "",
            f"  Reversibility : {self.reversibility.value.upper()}",
        ]
        if self.rollback_path:
            lines.append(f"  Rollback path : {self.rollback_path}")
        else:
            lines.append("  Rollback path : NONE — this action cannot be undone")

        lines += [
            "",
            "  WHAT CANNOT BE UNDONE:",
        ]
        for effect in self.irreversible_effects:
            lines.append(f"    • {effect}")

        lines += [
            "",
            f"  BLAST RADIUS  : {self.blast_radius.summary()}",
        ]
        if self.blast_radius.description:
            lines.append(f"  Detail        : {self.blast_radius.description}")

        lines += [
            "",
            f"  Dry-run done  : {'YES' if self.dry_run_completed else 'NO — CAUTION'}",
        ]
        if self.dry_run_summary:
            lines.append(f"  Dry-run result: {self.dry_run_summary}")

        lines += [
            "",
            f"  Timeout       : {self.timeout_seconds}s — NO RESPONSE = ABORT",
            "=" * 60,
            "  Type 'approve' to proceed, anything else to abort.",
            "=" * 60,
        ]
        return "\n".join(lines)

### 3b. Claude-powered reversibility classifier

The agent calls Claude to classify an action before deciding whether a human gate is needed at all.

In [ ]:
CLASSIFICATION_TOOL = [
    {
        "name": "classify_action_reversibility",
        "description": (
            "Classify whether an agent action is reversible, recoverable, or permanent. "
            "Permanent actions require human approval with a full preflight report."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "reversibility": {
                    "type": "string",
                    "enum": ["reversible", "recoverable", "permanent"],
                    "description": (
                        "reversible: trivially undone (rename, toggle setting). "
                        "recoverable: rollback exists but requires explicit steps (restore from backup, replay log). "
                        "permanent: no undo path (send email, delete without backup, charge a card, publish publicly)."
                    ),
                },
                "reasoning": {
                    "type": "string",
                    "description": "Brief justification for the classification.",
                },
                "irreversible_effects": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of specific effects that cannot be undone, even if rollback exists.",
                },
                "suggested_rollback_path": {
                    "type": "string",
                    "description": "Steps to undo the action. Leave blank if no rollback is possible.",
                },
            },
            "required": ["reversibility", "reasoning", "irreversible_effects"],
        },
    }
]


def classify_reversibility(action_name: str, action_description: str) -> dict:
    """Ask Claude to classify the reversibility of an action.

    Returns the tool input dict with reversibility, reasoning, etc.
    """
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=512,
        tools=CLASSIFICATION_TOOL,
        tool_choice={"type": "auto"},
        messages=[
            {
                "role": "user",
                "content": (
                    f"Classify the reversibility of this agent action:\n\n"
                    f"Action name: {action_name}\n"
                    f"Description: {action_description}\n\n"
                    "Use the classify_action_reversibility tool."
                ),
            }
        ],
    )

    for block in message.content:
        if block.type == "tool_use" and block.name == "classify_action_reversibility":
            return block.input

    # Fallback: treat as permanent if classification fails
    return {
        "reversibility": "permanent",
        "reasoning": "Classification failed; defaulting to permanent for safety.",
        "irreversible_effects": ["Unknown — classification failed"],
        "suggested_rollback_path": "",
    }

### 3c. The gate class with timeout-abort

In [ ]:
class ApprovalAbortedError(Exception):
    """Raised when the human denies approval or the gate times out."""


class IrreversibleActionGate:
    """Gate that enforces preflight checks before asking a human to approve an action.

    Usage:
        gate = IrreversibleActionGate(timeout_seconds=60)
        await gate.request_approval(
            action_name="send_bulk_email",
            action_description="...",
            blast_radius=BlastRadius(...),
            dry_run_fn=my_dry_run,
        )
        # Only reaches here if human typed 'approve' within timeout.
    """

    def __init__(self, timeout_seconds: int = 60):
        self.timeout_seconds = timeout_seconds

    async def request_approval(
        self,
        action_name: str,
        action_description: str,
        blast_radius: BlastRadius,
        dry_run_fn=None,          # async callable → str summary
        rollback_path: Optional[str] = None,
        _mock_response: Optional[str] = None,  # for notebook demos only
    ) -> None:
        """Run preflight, then present a structured approval request.

        Raises ApprovalAbortedError if the human denies or times out.
        """
        print(f"\n[Gate] Classifying reversibility of '{action_name}'...")
        classification = classify_reversibility(action_name, action_description)
        reversibility = Reversibility(classification["reversibility"])
        print(f"[Gate] Classification: {reversibility.value.upper()}")
        print(f"[Gate] Reasoning: {classification['reasoning']}")

        if reversibility == Reversibility.REVERSIBLE:
            print("[Gate] Action is reversible — no human approval required.")
            return

        # --- Preflight: run dry-run BEFORE presenting the gate ---
        dry_run_completed = False
        dry_run_summary = None
        if dry_run_fn is not None:
            print("[Gate] Running dry-run...")
            try:
                dry_run_summary = await dry_run_fn()
                dry_run_completed = True
                print(f"[Gate] Dry-run complete: {dry_run_summary}")
            except Exception as exc:
                print(f"[Gate] Dry-run FAILED: {exc}")
                raise ApprovalAbortedError(
                    f"Aborting: dry-run failed with error: {exc}"
                ) from exc
        else:
            print("[Gate] WARNING: No dry-run function provided. Proceeding without preview.")

        # --- Build the structured approval request ---
        suggested_rollback = classification.get("suggested_rollback_path") or rollback_path
        request = ApprovalRequest(
            action_name=action_name,
            action_description=action_description,
            reversibility=reversibility,
            irreversible_effects=classification.get("irreversible_effects", []),
            blast_radius=blast_radius,
            dry_run_completed=dry_run_completed,
            dry_run_summary=dry_run_summary,
            rollback_path=suggested_rollback if suggested_rollback else None,
            timeout_seconds=self.timeout_seconds,
        )

        print("\n" + request.render())

        # --- Wait for human response with hard timeout ---
        if _mock_response is not None:
            # In notebooks: skip interactive input and use the mock value
            human_input = _mock_response
            print(f"[Demo] Simulated response: '{human_input}'")
        else:
            try:
                human_input = await asyncio.wait_for(
                    asyncio.get_event_loop().run_in_executor(None, input, "\nYour decision: "),
                    timeout=self.timeout_seconds,
                )
            except asyncio.TimeoutError:
                raise ApprovalAbortedError(
                    f"Approval timed out after {self.timeout_seconds}s — action aborted (safe default)."
                )

        if human_input.strip().lower() != "approve":
            raise ApprovalAbortedError(
                f"Human denied approval (response: '{human_input}') — action aborted."
            )

        print("[Gate] Approval confirmed. Proceeding.")

## Part 4 — Three concrete examples

### Example 1: Sending a bulk email to 10,000 users

Emails are a canonical permanent action: once sent, they cannot be recalled.

In [ ]:
async def dry_run_bulk_email() -> str:
    """Simulate rendering all emails without sending."""
    # In production: call your email provider's preview/validate endpoint
    await asyncio.sleep(0.1)  # simulate I/O
    return (
        "Rendered 10,000 email bodies (sample: 'Hi Alice, your June invoice...'). "
        "No template errors. Estimated delivery time 4m 12s. "
        "Unsubscribe link verified present in all messages."
    )


async def example_bulk_email():
    gate = IrreversibleActionGate(timeout_seconds=60)

    try:
        await gate.request_approval(
            action_name="send_bulk_email",
            action_description=(
                "Send a 'June billing summary' email to 10,000 active users via SendGrid. "
                "Each email is personalised with the recipient's invoice total and account status."
            ),
            blast_radius=BlastRadius(
                users_impacted=10_000,
                description="All active users with a June invoice. Includes 1,240 users on payment plans.",
            ),
            dry_run_fn=dry_run_bulk_email,
            rollback_path=None,  # no rollback for sent email
            _mock_response="approve",  # notebook demo: skip interactive input
        )
        # Only reached if human approved
        print("\nACTION EXECUTED: 10,000 emails dispatched via SendGrid at 2026-06-21T08:00:00Z.")

    except ApprovalAbortedError as exc:
        print(f"\nACTION ABORTED: {exc}")


await example_bulk_email()


[Gate] Classifying reversibility of 'send_bulk_email'...
[Gate] Classification: PERMANENT
[Gate] Reasoning: Sending emails is a permanent action. Once emails are delivered to recipients' inboxes, they cannot be recalled or deleted by the sender. Even if the email service allows scheduling, cancellation is only possible before delivery.
[Gate] Running dry-run...
[Gate] Dry-run complete: Rendered 10,000 email bodies (sample: 'Hi Alice, your June invoice...'). No template errors. Estimated delivery time 4m 12s. Unsubscribe link verified present in all messages.

  APPROVAL REQUIRED — IRREVERSIBLE ACTION
  Action      : send_bulk_email
  What happens: Send a 'June billing summary' email to 10,000 active users via SendGrid. Each email is personalised with the recipient's invoice total and account status.

  Reversibility : PERMANENT
  Rollback path : NONE — this action cannot be undone

  WHAT CANNOT BE UNDONE:
    • Recipients will receive the email immediately upon sending
    • Email co

### Example 2: Deleting a database record

Deletion from a database with a recent backup is `recoverable` — but there are still permanent side effects the human must understand (audit log, cascades, time cost of restore).

In [ ]:
async def dry_run_db_delete() -> str:
    """Run EXPLAIN / SELECT COUNT(*) — never the real DELETE."""
    await asyncio.sleep(0.1)
    return (
        "DELETE FROM orders WHERE status='cancelled' AND created_at < '2025-01-01' "
        "would affect 847 rows. "
        "Cascade check: 0 rows in order_items (already cleaned). "
        "Foreign key constraint on invoices: 12 invoice rows would cascade-delete. "
        "Last backup: 2026-06-21T06:00:00Z (2h ago). Restore estimated at 45 minutes."
    )


async def example_db_delete():
    gate = IrreversibleActionGate(timeout_seconds=60)

    try:
        await gate.request_approval(
            action_name="delete_database_records",
            action_description=(
                "DELETE FROM orders WHERE status='cancelled' AND created_at < '2025-01-01'. "
                "Removes 847 cancelled orders older than 18 months."
            ),
            blast_radius=BlastRadius(
                records_affected=847,
                description="847 cancelled orders + 12 cascaded invoice rows. Zero active or pending orders in scope.",
            ),
            dry_run_fn=dry_run_db_delete,
            rollback_path=(
                "Restore from snapshot prod-db-2026-06-21T06:00:00Z. "
                "Estimated 45 min downtime. Requires on-call DBA sign-off."
            ),
            _mock_response="approve",
        )
        print("\nACTION EXECUTED: 847 orders deleted. 12 invoice rows cascade-deleted. Transaction committed.")

    except ApprovalAbortedError as exc:
        print(f"\nACTION ABORTED: {exc}")


await example_db_delete()


[Gate] Classifying reversibility of 'delete_database_records'...
[Gate] Classification: RECOVERABLE
[Gate] Reasoning: Database records can be recovered from backups if they exist and are recent enough. However, the deletion itself and any cascading effects on related records happen immediately and require explicit recovery steps.
[Gate] Running dry-run...
[Gate] Dry-run complete: DELETE FROM orders WHERE status='cancelled' AND created_at < '2025-01-01' would affect 847 rows. Cascade check: 0 rows in order_items (already cleaned). Foreign key constraint on invoices: 12 invoice rows would cascade-delete. Last backup: 2026-06-21T06:00:00Z (2h ago). Restore estimated at 45 minutes.

  APPROVAL REQUIRED — IRREVERSIBLE ACTION
  Action      : delete_database_records
  What happens: DELETE FROM orders WHERE status='cancelled' AND created_at < '2025-01-01'. Removes 847 cancelled orders older than 18 months.

  Reversibility : RECOVERABLE
  Rollback path : Restore from snapshot prod-db-2026-06-

### Example 3: Deploying to production

A production deploy is recoverable (rollback to previous image), but the window between deploy and rollback carries real risk. The human needs to know the rollback procedure *before* they approve.

In [ ]:
async def dry_run_production_deploy() -> str:
    """Deploy to staging, run smoke tests, return a summary."""
    await asyncio.sleep(0.1)
    return (
        "Staging deploy of api-service:v2.4.1 completed in 87s. "
        "Health checks: 12/12 passed. No DB migrations in this release. "
        "p99 latency on staging: 143ms (was 151ms). "
        "Zero errors in smoke test suite (47 tests)."
    )


async def example_production_deploy():
    gate = IrreversibleActionGate(timeout_seconds=60)

    try:
        await gate.request_approval(
            action_name="deploy_to_production",
            action_description=(
                "Deploy container image api-service:v2.4.1 to production ECS cluster "
                "(3 availability zones, 12 tasks). Rolling deploy with 25% max-surge. "
                "Health check grace period: 30s."
            ),
            blast_radius=BlastRadius(
                users_impacted=10_000,
                estimated_cost_usd=2_400.0,
                description=(
                    "All users of the API during the 3-minute rolling window. "
                    "~$2,400/hr revenue impact if the deploy causes an outage."
                ),
            ),
            dry_run_fn=dry_run_production_deploy,
            rollback_path=(
                "Run `ecs deploy api-service --image api-service:v2.4.0`. "
                "Estimated rollback time: 3 minutes. Previous image is pinned and available."
            ),
            _mock_response="no",  # simulate denial to show abort path
        )
        print("\nACTION EXECUTED: Production deploy started. Monitoring CloudWatch for error rate spikes.")

    except ApprovalAbortedError as exc:
        print(f"\nACTION ABORTED: {exc}")


await example_production_deploy()


[Gate] Classifying reversibility of 'deploy_to_production'...
[Gate] Classification: RECOVERABLE
[Gate] Reasoning: A production deployment can be rolled back by redeploying the previous container image. However, any database migrations that run during the deployment may not be easily reversible, and there is a window of risk between deployment and potential rollback where errors or downtime can affect users.
[Gate] Running dry-run...
[Gate] Dry-run complete: Staging deploy of api-service:v2.4.1 completed in 87s. Health checks: 12/12 passed. No DB migrations in this release. p99 latency on staging: 143ms (was 151ms). Zero errors in smoke test suite (47 tests).

  APPROVAL REQUIRED — IRREVERSIBLE ACTION
  Action      : deploy_to_production
  What happens: Deploy container image api-service:v2.4.1 to production ECS cluster (3 availability zones, 12 tasks). Rolling deploy with 25% max-surge. Health check grace period: 30s.

  Reversibility : RECOVERABLE
  Rollback path : Run `ecs deploy a

## Part 5 — The timeout abort pattern in detail

The gate uses `asyncio.wait_for` to enforce a hard deadline. If no response arrives within `timeout_seconds`, it raises `ApprovalAbortedError` — it does **not** proceed.

This is the safe default: **silence = abort**, not silence = approve.

In [ ]:
timeout_pattern_explanation = """
The gate calls:

    human_input = await asyncio.wait_for(
        asyncio.get_event_loop().run_in_executor(None, input, "Your decision: "),
        timeout=self.timeout_seconds,
    )

If asyncio.TimeoutError fires:
    → raise ApprovalAbortedError("Approval timed out — action aborted (safe default).")

If human types anything other than 'approve':
    → raise ApprovalAbortedError("Human denied — action aborted.")

Only 'approve' (exact, case-insensitive) allows the action to proceed.

Why 'approve' and not 'y'?
  A full word requires intentional typing. Fat-fingering a confirmation
  prompt with 'y' (or accidentally pressing Enter) cannot trigger deployment.

Why abort on timeout, not proceed?
  The cost of a false negative (missing an approval window) is recoverable —
  just re-run the agent. The cost of a false positive (unattended agent
  deletes production data while the operator is away) is not.
"""

print("=== Timeout abort pattern ===")
print(timeout_pattern_explanation)

=== Timeout abort pattern ===

The gate calls:

    human_input = await asyncio.wait_for(
        asyncio.get_event_loop().run_in_executor(None, input, "Your decision: "),
        timeout=self.timeout_seconds,
    )

If asyncio.TimeoutError fires:
    → raise ApprovalAbortedError("Approval timed out — action aborted (safe default).")

If human types anything other than 'approve':
    → raise ApprovalAbortedError("Human denied — action aborted.")

Only 'approve' (exact, case-insensitive) allows the action to proceed.

Why 'approve' and not 'y'?
  A full word requires intentional typing. Fat-fingering a confirmation
  prompt with 'y' (or accidentally pressing Enter) cannot trigger deployment.

Why abort on timeout, not proceed?
  The cost of a false negative (missing an approval window) is recoverable —
  just re-run the agent. The cost of a false positive (unattended agent
  deletes production data while the operator is away) is not.


## Summary — The approval gate checklist

Before your agent presents a human-in-the-loop gate for any irreversible action, verify all of the following:

| Step | Question | Why it matters |
|------|----------|----------------|
| 1 | Is the action `reversible`, `recoverable`, or `permanent`? | Reversible actions don't need gates; skipping the gate for permanent ones is dangerous |
| 2 | Was a dry-run run before the gate? | Catches bugs before they touch production data |
| 3 | Is the blast radius quantified? | The human cannot assess risk without numbers |
| 4 | Is the rollback path documented, or explicitly `None`? | The human must know whether they can undo |
| 5 | Are the irreversible effects listed, even for recoverable actions? | Rollback ≠ zero harm; side effects still happen |
| 6 | Does timeout → abort, not timeout → proceed? | Safe default: silence must never be interpreted as consent |
| 7 | Is the approval word unambiguous (e.g., `approve`, not `y`)? | Accidental confirmations must be prevented |

The naive gate skips all seven. The `IrreversibleActionGate` enforces all seven.

### When to use each reversibility classification

- **`reversible`**: No gate needed. Agent proceeds automatically. Example: rename a file, toggle a feature flag.
- **`recoverable`**: Gate required. Show rollback path and blast radius. Example: delete DB records with recent backup, deploy with rollback image.
- **`permanent`**: Gate required. Explicitly surface that no rollback exists. Example: send email, charge credit card, publish public post.